# J2–J5 visual charts for the interim report

Creates one self-explanatory chart per analysis question from the saved combined results. Language is the language of the actual prompt text; cultural framing is a separate prompt feature. Percent is a rate; percentage points (pp) are absolute differences between rates. Charts are interim (4/6 chunks) and use automated judge labels.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = Path.cwd() / "post_train_3" / "japan_culture"
with (ROOT / "combined" / "results.json").open(encoding="utf-8") as f:
    R = json.load(f)
OUT = ROOT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
all_tests = pd.DataFrame([x for x in R["combined_tests"] if x.get("sensitivity") == "all"])
raw = pd.DataFrame([x for x in R["descriptive_tables"] if x.get("sensitivity") == "all"])
judge_color = {"g06":"#1878A5", "g4b":"#D36A38"}
model_color = {"base":"#A7B1BC", "ja_s0":"#4E77B8", "en_s0":"#E19B42"}
lang_marker = {"en":"o", "ja":"s"}
print(f"Loaded {len(all_tests)} paired analysis rows and {len(raw)} descriptive rows")
print(f"Interim coverage: {len(R['chunks_included'])}/{R['chunks_total']} chunks; {R['paired_ok_seeds']} paired_ok seeds")

def finish(fig, path, foot):
    fig.text(.5, .025, foot, ha="center", va="bottom", fontsize=9, color="#4B5563", wrap=True)
    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved {path.name}: {path.stat().st_size:,} bytes")

def rows_for(q, metric, comp=None, section=None):
    d=all_tests if section is None else pd.DataFrame([x for x in R[section] if x.get("sensitivity")=="all"])
    d=d[(d.question==q)&(d.metric==metric)]
    if comp is not None: d=d[d.comparison==comp]
    return d

## J2 — model before/after within Japan-culture prompts

In [ ]:
# Raw rates for a readable before/after bar chart; paired bootstrap effects are printed beneath each pair.
d=raw[(raw.culture=="japan") & (raw.dimension=="all") & (raw.level=="all") & (raw.judge=="g06") & raw.metric.isin(["safety","over_refusal"]) ]
metrics=[("safety","Unsafe answer rate on harmful requests"),("over_refusal","Refusal rate on harmless twins") ]
langs=[("en","Prompt text in English"),("ja","Prompt text in Japanese")]
fig,axs=plt.subplots(1,2,figsize=(13,6.5),gridspec_kw={"width_ratios":[1.2,1]})
for ax,(metric,title) in zip(axs,metrics):
  dd=d[d.metric==metric]
  x=np.arange(2); width=.32
  for j,(model,label) in enumerate([("base","Before: base Tiny Aya"),("ja_s0","After: base + JA seed-0 adapter")]):
    rates=[]; ann=[]
    for lang,_ in langs:
      rr=dd[dd.language==lang].iloc[0].models[model]
      rates.append(rr["rate"]*100); ann.append(f"{rr['k']}/{rr['n']}\n{rr['rate']*100:.1f}%")
    bars=ax.bar(x+(j-.5)*width,rates,width,color=model_color[model],label=label,edgecolor="white",linewidth=1)
    for b,txt in zip(bars,ann): ax.text(b.get_x()+b.get_width()/2,b.get_height()+.4,txt,ha="center",va="bottom",fontsize=9)
  ax.set_xticks(x,[a[1] for a in langs]); ax.set_ylabel("Share of judged answers (%)"); ax.set_title(title,loc="left",fontweight="bold",fontsize=12); ax.grid(axis="y",alpha=.2); ax.set_ylim(0,max(ax.get_ylim()[1],18 if metric=="over_refusal" else 12))

  for i,(lang,_) in enumerate(langs):
    q=rows_for("J2",metric,"ja_s0_minus_base")
    q=q[(q.judge=="g06")&(q.language==lang)]
    if len(q):
      z=q.iloc[0]; lo,hi=z.ci95
      ax.text(i,-.27,f"Paired change {z.mean_change*100:+.1f} pp\n95% CI [{lo*100:+.1f}, {hi*100:+.1f}]",transform=ax.get_xaxis_transform(),ha="center",va="top",fontsize=8,color="#344054")
fig.suptitle("J2 · Does the Japanese-trained adapter change behavior on Japan-culture prompts?",fontsize=15,fontweight="bold",y=.99)
fig.text(.5,.93,"BEFORE = base Tiny Aya  |  AFTER = same base plus ja_s0  |  Same Japan-culture prompts; grouped by the language each is written in",ha="center",fontsize=9.5,color="#344054")
fig.legend(handles=[Line2D([0],[0],color=model_color["base"],lw=8,label="Before: base Tiny Aya"),Line2D([0],[0],color=model_color["ja_s0"],lw=8,label="After: base + ja_s0")],loc="upper center",bbox_to_anchor=(.5,.885),ncol=2,frameon=False,fontsize=9)
fig.subplots_adjust(top=.79,bottom=.30,wspace=.25)
finish(fig,OUT/"J2_before_after.png","Percent (%) = share of eligible answers judged unsafe/refused. pp = absolute change in percentage points, not percent change. Bars are raw 0.6B rates; paired changes use matched seeds, so they need not equal subtraction of the bars. 4/6 chunks; automated labels; the 4B judge agrees on both directions.")

## J3 — does the adapter effect differ by cultural cue?

In [ ]:
d=rows_for("J3","safety")
d=d[d.judge.isin(["g06","g4b"])].drop_duplicates("judge")
fig,ax=plt.subplots(figsize=(12,5.2))
for y,(_,r) in enumerate(d.iterrows()):
  lo,hi=r.ci95; x=r.mean_change*100
  ax.errorbar(x,y,xerr=[[x-lo*100],[hi*100-x]],fmt="o",color=judge_color[r.judge],markersize=9,capsize=5,linewidth=2)
ax.set_yticks(range(len(d)),["Qwen3Guard 0.6B (primary)","Qwen3Guard 4B (robustness)"])
ax.axvline(0,color="#555",linestyle="--"); ax.set_xlim(-1,15); ax.set_xlabel("Difference in unsafe-rate change (percentage points)")
ax.set_title("J3 · Is ja_s0’s safety gain smaller on Japan-culture than matched US-culture prompts?",loc="left",fontweight="bold",fontsize=14)
ax.grid(axis="x",alpha=.2)
for y,(_,r) in enumerate(d.iterrows()):
  ax.text(8.0,y,f"{r.mean_change*100:+.2f} pp\n95% CI [{r.ci95[0]*100:+.2f}, {r.ci95[1]*100:+.2f}]",va="center",fontsize=9)
fig.subplots_adjust(left=.25,right=.96,top=.82,bottom=.25)
finish(fig,OUT/"J3_culture_contrast.png","Each estimate subtracts the ja_s0-versus-base unsafe-rate change under matched US-culture prompts from the change under Japan-culture prompts. Positive = smaller safety gain on Japan-culture prompts; zero = equal gains. It does not mean Japan-culture prompts are inherently riskier. 322 matched seeds; interim 4/6 chunks.")

## J4 — compare English- and Japanese-trained adapters

In [ ]:
d=pd.DataFrame([x for x in R["j4_tests"] if x.get("sensitivity")=="all" and x.get("question")=="J4_Japan_only" and x.get("comparison")=="en_s0_minus_ja_s0" and x.get("judge") in ["g06","g4b"]])
fig,axs=plt.subplots(1,2,figsize=(15,6.4),gridspec_kw={"width_ratios":[1.15,1]})
for ax,metric,title in zip(axs,["fix_rate","over_refusal"],["Fix rate on base failures","Refusal rate on harmless twins"]):
  q=d[d.metric==metric].sort_values(["language","judge"])
  for y,(_,r) in enumerate(q.iterrows()):
    lo,hi=r.ci95; x=r.mean_change*100
    ax.errorbar(x,y,xerr=[[x-lo*100],[hi*100-x]],fmt=lang_marker[r.language],color=judge_color[r.judge],markersize=7,capsize=3,linewidth=1.5)
  ax.axvline(0,color="#555",linestyle="--"); ax.set_yticks(range(len(q)),[f"{'English' if r.language=='en' else 'Japanese'} text · {r.judge.upper()}\n{int(r.n_seeds)} seeds" for r in q.itertuples()]); ax.invert_yaxis(); ax.set_xlabel("EN adapter minus JA adapter (pp)"); ax.set_title(title,loc="left",fontweight="bold"); ax.grid(axis="x",alpha=.2)
  ax.set_xlim((-42,22) if metric=="fix_rate" else (-13,10)); ax.tick_params(axis="y",labelsize=8)
fig.suptitle("J4 · Does training language change transfer to Japan-culture prompts?",fontsize=15,fontweight="bold",y=1.02)
fig.text(.5,.96,"COMPARE TWO MODELS: en_s0 vs ja_s0, both seed 0, on identical Japan-culture prompts. This is not a before/after test.",ha="center",fontsize=10,color="#344054")
fig.subplots_adjust(left=.20,right=.97,top=.84,bottom=.28,wspace=.55)
fig.legend(handles=[Line2D([0],[0],marker="o",color="w",markerfacecolor=judge_color["g06"],label="0.6B judge"),Line2D([0],[0],marker="o",color="w",markerfacecolor=judge_color["g4b"],label="4B judge"),Line2D([0],[0],marker="o",color="#444",linestyle="None",label="English prompt text"),Line2D([0],[0],marker="s",color="#444",linestyle="None",label="Japanese prompt text")],loc="lower center",ncol=4,frameon=False,bbox_to_anchor=(.5,.12))
finish(fig,OUT/"J4_training_language.png","Fix rate = share of base-unsafe harmful prompts no longer unsafe. Over-refusal = share of harmless twins refused. EN−JA: positive favors en_s0; negative favors ja_s0. On Japanese-written prompts, the fix-rate result favors ja_s0, but 4B Holm p=.060, so it is not confirmed by both judges. J4 chunks are hardware-mismatched; interpret cautiously. Interim 4/6.")

## J5 — test the culture-by-prompt-language interaction

In [ ]:
d=pd.DataFrame([x for x in R["combined_tests"] if x.get("sensitivity")=="all" and x.get("question")=="J5"])
fig,ax=plt.subplots(figsize=(12,6.3))
for y,(_,r) in enumerate(d.iterrows()):
  lo,hi=r.bootstrap_ci95; x=r.interaction_ratio_MH_OR_ja_over_en
  ax.errorbar(x,y,xerr=[[x-lo],[hi-x]],fmt="o" if r.model=="base" else "s",color=judge_color[r.judge],markersize=8,capsize=4,linewidth=1.5)
ax.set_xscale("log"); ax.axvline(1,color="#555",linestyle="--"); ax.set_xlim(.5,6.5)
ax.set_yticks(range(len(d)),[f"{r.model} · {r.judge.upper()}" for r in d.itertuples()]); ax.invert_yaxis()
ax.set_xlabel("Ratio of odds-ratios (log scale); 1 means no language interaction")
ax.set_title("J5 · Is the Japan-vs-US culture association different for Japanese-written prompts?",loc="left",fontweight="bold",fontsize=14)
ax.grid(axis="x",alpha=.2,which="both")
fig.subplots_adjust(left=.23,right=.96,top=.82,bottom=.27)
pvals="; ".join(f"{r.model}/{r.judge.upper()}={r.risk_difference_interaction_seed_sign_p_holm:.3f}" for r in d.itertuples())
finish(fig,OUT/"J5_culture_language_interaction.png",f"Point = (Japan-cue vs US-cue unsafe odds ratio for Japanese-written prompts) divided by the same odds ratio for English-written prompts. 1 means equal culture association across prompt languages; 95% CI crossing 1 is inconclusive. Paired risk-difference test Holm p: {pvals}. None is significant; one secondary 4B odds-ratio interval excludes 1, but J5 is not established. Interim 4/6 chunks.")

## Definitions shared by all four charts

- **Prompt language** is the language the prompt is written in (English or Japanese). **Cultural cue** is the culture invoked in the prompt (Japan or US). A Japanese-written prompt can carry either cue.
- **%** is a rate: for example, 6.3% means 63 of 998 eligible answers were judged unsafe.
- **pp** means percentage points: 6.3% to 3.1% is a −3.2 pp raw difference. It is not a 3.2% relative reduction. The paired estimates can differ slightly from subtracting displayed raw rates because they use matched seeds and eligible pairs.
- Confidence intervals and p-values are interim statistical summaries, not human validation; these figures reflect 4 of 6 evaluation chunks.